# 03 — Statistika Inferensial (Uji Perbedaan)
Menjawab Rumusan Masalah 2 dan 3: menguji perbedaan jumlah dan intensitas titik panas antara tahun El Niño 2023 dan Non-El Niño, serta antar pulau.
Uji non-parametrik (Mann-Whitney U, Kruskal-Wallis, Chi-square) dipilih karena data tidak berdistribusi normal (skewness & kurtosis jauh di luar ±2).
Karena n sangat besar, selain p-value juga dilaporkan **ukuran efek**.

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu, kruskal, chi2_contingency

data = pd.read_csv("data_analisis.csv")
harian = pd.read_csv("harian_pulau.csv").dropna(subset=["jumlah_titik_panas"])
ALPHA = 0.05

In [ ]:
# Cek Kelengkapan Data
print("data_analisis :", data.shape, "| harus (515450, 13)")
print("harian_pulau  :", harian.shape, "| harus (12516, 7) setelah 266 baris missing dibuang")
print(data["kelompok_tahun"].value_counts())
assert len(data) == 515450, "data_analisis.csv TIDAK LENGKAP"
assert len(harian) == 12516, "harian_pulau.csv TIDAK LENGKAP"
print("Data lengkap, lanjutkan.")

In [ ]:
# RM2: El Nino 2023 vs Non-El Nino (Mann-Whitney U) + ukuran efek r
def uji_mann_whitney(df, kolom):
    a = df.loc[df["kelompok_tahun"] == "El Nino 2023", kolom]
    b = df.loc[df["kelompok_tahun"] == "Non-El Nino", kolom]
    u, p = mannwhitneyu(a, b, alternative="two-sided")
    n1, n2 = len(a), len(b)
    z = (u - n1 * n2 / 2) / np.sqrt(n1 * n2 * (n1 + n2 + 1) / 12)
    return {"variabel": kolom, "median El Nino": a.median(), "median Non": b.median(),
            "U": u, "p-value": p, "r (efek)": abs(z) / np.sqrt(n1 + n2),
            "keputusan": "Tolak H0" if p < ALPHA else "Gagal tolak H0"}

hasil_mw = pd.DataFrame([uji_mann_whitney(harian, "jumlah_titik_panas"), uji_mann_whitney(data, "frp")])
hasil_mw

In [ ]:
# RM3: perbedaan antar pulau (Kruskal-Wallis) + ukuran efek epsilon-kuadrat
def uji_kruskal(df, kolom):
    kelompok = [g[kolom].values for _, g in df.dropna(subset=["pulau"]).groupby("pulau")]
    h, p = kruskal(*kelompok)
    n = sum(len(k) for k in kelompok)
    return {"variabel": kolom, "H": h, "p-value": p, "epsilon^2 (efek)": h / (n - 1),
            "keputusan": "Tolak H0" if p < ALPHA else "Gagal tolak H0"}

hasil_kw = pd.DataFrame([uji_kruskal(harian, "jumlah_titik_panas"), uji_kruskal(data, "frp")])
hasil_kw

In [ ]:
# Hubungan pulau x kelompok tahun (Chi-square) + Cramer's V
tabel = pd.crosstab(data["pulau"], data["kelompok_tahun"])
chi2, p, dof, _ = chi2_contingency(tabel)
cramer_v = np.sqrt(chi2 / (tabel.values.sum() * (min(tabel.shape) - 1)))
print(tabel)
print(f"\nChi-square = {chi2:,.2f} | df = {dof} | p-value = {p:.3g} | Cramer's V = {cramer_v:.3f}")

**Acuan ukuran efek:** r ≈ 0,1 kecil · 0,3 sedang · 0,5 besar | ε² ≈ 0,01 kecil · 0,08 sedang · 0,26 besar | Cramér's V (df=6 → df* = 1) ≈ 0,1 kecil · 0,3 sedang · 0,5 besar.